# Notebook 06 — Mini Capstone: Smart FAQ Finder (sample solution)

> **Easiest way to run this: Google Colab — nothing to install.**
> Go to https://colab.research.google.com → **File > Upload notebook** → choose this file.
> Prefer your own computer? Lesson 1 shows the VS Code and local-Jupyter paths too.

This is the **sample solution** for the mini capstone in Lesson 11. Try building yours first
from the lesson's checklist; come back here to check your work — no shame in peeking.

**The task:** a pizza shop has a list of FAQ questions and answers. Build a tool where a
customer types a question *in their own words* and gets the right answer — even when the
wording is completely different from the stored question.

In [ ]:
%pip install -q sentence-transformers chromadb
print("Ready.")

## Step 1 — The FAQ (ten question → answer pairs)

In [ ]:
faq = [
    ("What are your opening hours?",
     "We're open 11am to 11pm every day, including weekends."),
    ("Do you offer vegan options?",
     "Yes — we have plant-based cheese and several vegetable-only pizzas."),
    ("Do you deliver?",
     "We deliver free within 5 km; a small fee applies beyond that."),
    ("How much is a large pizza?",
     "A large pizza starts at $14, before any extra toppings."),
    ("Can I book a table for a group?",
     "Yes, call us to reserve a table for groups of six or more."),
    ("Do you have gluten-free crust?",
     "We offer a gluten-free crust on any pizza for a small extra charge."),
    ("What payment methods do you accept?",
     "We take cash, all major cards, and the usual mobile wallets."),
    ("Is there parking nearby?",
     "There's a free public car park right behind the restaurant."),
    ("Do you cater for parties?",
     "Yes, we offer party platters and bulk orders with a day's notice."),
    ("How spicy is the hot pizza?",
     "Our spicy pizza is medium-hot; we can make it milder on request."),
]
questions = [q for q, a in faq]
answers   = [a for q, a in faq]
print(f"{len(faq)} FAQ pairs loaded.")

## Step 2 — Embed the FAQ questions into a store

We store the **questions** (that's what a customer's query should match), and keep each
answer alongside as metadata so we can return it.

In [ ]:
import chromadb
from chromadb.utils import embedding_functions

minilm_ef = embedding_functions.SentenceTransformerEmbeddingFunction(
    model_name="all-MiniLM-L6-v2"
)
client = chromadb.PersistentClient(path="./faq_store")
try:
    client.delete_collection("faq")
except Exception:
    pass
faq_collection = client.create_collection(
    name="faq",
    embedding_function=minilm_ef,
    metadata={"hnsw:space": "cosine"},
)
faq_collection.add(
    ids=[f"faq_{i}" for i in range(len(faq))],
    documents=questions,
    metadatas=[{"answer": a} for a in answers],
)
print(f"Indexed {faq_collection.count()} FAQ questions.")

## Step 3 — The `ask()` function

Find the closest stored question, then return its answer.

In [ ]:
def ask(question):
    result = faq_collection.query(query_texts=[question], n_results=1)
    matched_question = result["documents"][0][0]
    answer = result["metadatas"][0][0]["answer"]
    score = 1 - result["distances"][0][0]
    print(f"You asked : {question}")
    print(f"Matched   : {matched_question}  (similarity {score:.3f})")
    print(f"Answer    : {answer}\n")

ask("Are there plant-based choices on the menu?")

## Step 4 — Three reworded questions

None of these use the same words as the stored questions, yet each finds the right answer.

In [ ]:
ask("Until what time can I order tonight?")     # -> opening hours
ask("Can you bring it to my house?")            # -> delivery
ask("Where do I leave the car?")                # -> parking

## Recap

- A FAQ finder is the same pattern as semantic search: embed the questions, match a new
  question by meaning, return the stored answer.
- Storing the answer as metadata lets the store hand it back directly.
- "plant-based" matches "vegan", "leave the car" matches "parking" — because the model
  compares meaning, not words. That's the whole idea of the course, working for you.

You've now built two things end to end. 